# Extension Step 4 — GNN DAGNN Task Classifier v3 (Difference Fusion, small projector)

Trains a `DAGClassifier` with **difference fusion** and a compact projector on task graph realizations.

### Architecture evolution

| Component | v1 (concat) | v2 (diff, 256→256) | v3 (diff, 256→hidden_dim) |
|---|---|---|---|
| Fusion | `concat([t, v]) → Linear(512→256)` | `(t−v) → Linear(256→256)` | `(t−v) → Linear(256→32)` |
| `input_proj` | `Linear(256→128)` | `Linear(256→32)` | **removed** |
| `hidden_dim` | 128 | 32 | 32 |
| `num_layers` | 2 | 1 | 1 |
| Approx params | ~328K | ~80K | ~11K |

**Key change v2→v3**: projector maps directly into the DAGNN hidden space (`256→hidden_dim`),
removing the redundant `input_proj`. Dominant overfitting source (65K projector params) cut to ~8K.

**Semantic motivation of `t - v`**: in the EgoVLP joint space, `t - v` measures the gap
between the expected step description (`t`) and the observed visual execution (`v`).
Correct execution: `t ≈ v → t - v ≈ 0`. Error: `t - v` points in a characteristic direction.
Unmatched nodes (`v = 0`): `t - 0 = t`, so the text description is preserved as projector input.

**Prerequisites:**
- Step embeddings in `STEP_EMBEDDINGS_DIR` (output of Extension Step 1)
- Pre-fusion cache in `CACHE_DIR` (same cache as v1/v2 — no rebuild needed)
- Task graphs in `GRAPHS_DIR` (annotations submodule)
- EgoVLP checkpoint at `EGOVLP_CKPT`

**Output:** checkpoints + `results.csv` in `STEP4_OUTPUT_DIR`

In [ ]:
# ── 1. Mount Drive ────────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted.')

In [ ]:
# ── 2. Path constants + split mode ────────────────────────────────────────────

# CHOOSE YOUR SPLIT MODE
#   "video" -- CaptainCook4D recording-level split (video-stratified)
#   "recipe"   -- 16 train / 4 val / 4 test recipes, seed=42
#   "loo"      -- Leave-One-Out cross-validation (same protocol as B2)
SPLIT_MODE = "loo"

DRIVE_ROOT           = '/content/drive/MyDrive/AML_Project'
REPO_DIR             = '/content/code'
EGOVLP_REPO          = '/content/EgoVLP'

EGOVLP_CKPT          = f'{DRIVE_ROOT}/models/egovlp.pth'
STEP_EMBEDDINGS_DIR  = f'{DRIVE_ROOT}/step1/step_embeddings'
ANNOTATIONS_PATH     = f'{REPO_DIR}/annotations/annotation_json/complete_step_annotations.json'
GRAPHS_DIR           = f'{REPO_DIR}/annotations/task_graphs'
SPLITS_JSON          = f'{REPO_DIR}/extension/step1/combined_recordings.json'

# Pre-fusion cache — same as v1, no rebuild needed
CACHE_DIR            = f'{DRIVE_ROOT}/step3/cache_thr030'

# Output dir — separate from v1 to allow direct comparison
STEP4_OUTPUT_DIR     = f'{DRIVE_ROOT}/step4/results_diff_{SPLIT_MODE}'

LOCAL_EMBEDDINGS_DIR = '/content/step_embeddings'

print(f'split mode : {SPLIT_MODE}')
print(f'cache dir  : {CACHE_DIR}')
print(f'output dir : {STEP4_OUTPUT_DIR}')

In [ ]:
# ── 3. Clone repos ────────────────────────────────────────────────────────────
!git clone --recursive https://github.com/Laio95/aml-2025-mistake-detection.git {REPO_DIR}
!git clone https://github.com/showlab/EgoVLP.git {EGOVLP_REPO}

In [ ]:
# ── 4. Install dependencies ───────────────────────────────────────────────────
import torch, os
pt_version   = torch.__version__.split('+')[0]
cuda_version = torch.version.cuda
cuda_str     = f'cu{cuda_version.replace(".", "")}' if cuda_version else ''
os.environ['TORCH'] = pt_version
os.environ['CUDA']  = cuda_str
cuda_suffix = f'+{cuda_str}' if cuda_str else ''
print(f'PyTorch {pt_version}, CUDA {cuda_str or "not available"}')

!pip install torch-scatter torch-sparse -f https://data.pyg.org/whl/torch-${TORCH}{cuda_suffix}.html -q
!pip install decord pytorchvideo fvcore iopath torch-geometric -q
!pip install -r {REPO_DIR}/requirements.txt -q
print('Dependencies installed.')

In [ ]:
# ── 5. Copy step embeddings to local storage (faster than reading from Drive) ──
import os
os.makedirs(LOCAL_EMBEDDINGS_DIR, exist_ok=True)
!cp -r {STEP_EMBEDDINGS_DIR}/* {LOCAL_EMBEDDINGS_DIR}/
print(f'Copied: {len(os.listdir(LOCAL_EMBEDDINGS_DIR))} files → {LOCAL_EMBEDDINGS_DIR}')

In [ ]:
# ── 6. Verify pre-fusion cache ────────────────────────────────────────────────
# Same cache as v1 (sim_threshold=0.30). No rebuild needed.
# Cache stores raw text_feats and vis_feats; the DifferenceFusionProjector
# runs inside DAGClassifier.forward(), so the cache is architecture-agnostic.
import os, pathlib
cache_pt = list(pathlib.Path(CACHE_DIR).glob('*.pt')) if os.path.exists(CACHE_DIR) else []
if cache_pt:
    print(f'Cache ready: {len(cache_pt)} files in {CACHE_DIR}')
else:
    print(f'WARNING: cache not found at {CACHE_DIR}')
    print('Run gnn_task_classifier.ipynb first to build the cache.')

In [ ]:
# ── 7. Download ViT-B/16 checkpoint (required by EgoVLP FrozenInTime) ─────────
!mkdir -p {EGOVLP_REPO}/pretrained
!wget -q -nc -O {EGOVLP_REPO}/pretrained/jx_vit_base_p16_224-80ecf9dd.pth \
  https://github.com/rwightman/pytorch-image-models/releases/download/v0.1-vitjx/jx_vit_base_p16_224-80ecf9dd.pth
!ls -lh {EGOVLP_REPO}/pretrained/

In [ ]:
# ── 8. WandB login ────────────────────────────────────────────────────────────
!wandb login

## Smoke test — 3 epochs

Verify that `DAGClassifier` v2 (difference fusion, d_h=32, L=1) runs without errors.

**Estimated time:** ~1-2 min.

In [ ]:
SMOKE_OUTPUT_DIR = f'{DRIVE_ROOT}/step4/results_diff_smoke'

In [ ]:
import subprocess, sys, os
os.chdir('/content/code')

smoke_cmd = [
    sys.executable, '-m', 'extension.step4.train_dag_classifier',
    '--annotations_path',    ANNOTATIONS_PATH,
    '--step_embeddings_dir', LOCAL_EMBEDDINGS_DIR,
    '--graphs_dir',          GRAPHS_DIR,
    '--egovlp_repo',         EGOVLP_REPO,
    '--egovlp_ckpt',         EGOVLP_CKPT,
    '--cache_dir',           CACHE_DIR,
    '--output_dir',          SMOKE_OUTPUT_DIR,
    '--split_mode',          'loo',
    '--num_epochs',  '3',
    '--hidden_dim',  '32',
    '--num_layers',  '1',
    '--dropout',     '0.5',
    '--lr',          '1e-3',
    '--weight_decay','1e-4',
    '--batch_size',  '4',
    '--threshold',   '0.5',
    '--seed',        '42',
    '--num_workers', '2',
]
subprocess.run(smoke_cmd, check=True)

## Full training — 50 epochs

Trains the `DAGClassifier` v2 with the split selected in `SPLIT_MODE`.

### Supported configurations

| `SPLIT_MODE` | Protocol | Notes |
|---|---|---|
| `"loo"` | LOO 24-fold (same as B2) | direct comparison with TaskVerifier |
| `"video"` | 213/62/109 recordings | video split (recording-stratified) |
| `"recipe"` | 16/4/4 recipes, seed=42 | recipe-level split |

### Ablation baseline

To empirically observe the overfitting that still occurs with d_h=128 L=2 (difference fusion):
change `--hidden_dim 128 --num_layers 2` in the command below and use a separate output dir.

**Estimated time:** ~15-30 min on T4 for video/recipe; ~6-10 hours for LOO.

In [ ]:
import subprocess, sys, os
os.chdir('/content/code')

common = [
    '--annotations_path',    ANNOTATIONS_PATH,
    '--step_embeddings_dir', LOCAL_EMBEDDINGS_DIR,
    '--graphs_dir',          GRAPHS_DIR,
    '--egovlp_repo',         EGOVLP_REPO,
    '--egovlp_ckpt',         EGOVLP_CKPT,
    '--cache_dir',           CACHE_DIR,
    '--output_dir',          STEP4_OUTPUT_DIR,
    '--num_epochs',  '50',
    '--hidden_dim',  '32',
    '--num_layers',  '1',
    '--dropout',     '0.5',
    '--lr',          '1e-3',
    '--weight_decay','1e-4',
    '--batch_size',  '4',
    '--threshold',   '0.5',
    '--seed',        '42',
    '--num_workers', '2',
    '--enable_wandb',
    '--wandb_project', 'gnn_task_verification_diff_v3',
]

if SPLIT_MODE == 'loo':
    cmd = [sys.executable, '-m', 'extension.step4.train_dag_classifier',
           *common, '--split_mode', 'loo']
elif SPLIT_MODE == 'recipe':
    cmd = [sys.executable, '-m', 'extension.step4.train_dag_classifier',
           *common, '--split_mode', 'recipe', '--recipe_split_seed', '42']
else:  # official
    cmd = [sys.executable, '-m', 'extension.step4.train_dag_classifier',
           *common, '--split_mode', 'official', '--splits_json', SPLITS_JSON]

subprocess.run(cmd, check=True)

## Risultati

In [ ]:
# ── Read and print results.csv ──────────────────────────────────────────────
import pandas as pd

csv_path = f'{STEP4_OUTPUT_DIR}/results.csv'
df = pd.read_csv(csv_path)
print(df.to_string(index=False))